In [1]:
import pandas as pd
import glob

In [ ]:
csv_files = glob.glob('api_data_aadhar_biometric/api_data_aadhar_biometric_*.csv') + \
              glob.glob('api_data_aadhar_demographic/api_data_aadhar_demographic_*.csv') + \
              glob.glob('api_data_aadhar_enrolment/api_data_aadhar_enrolment_*.csv')
dfs = []
for file in csv_files:
    try:
        df = pd.read_csv(file)
        df['source_file'] = file 
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {file}: {e}")

if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"Successfully combined {len(dfs)} CSV files into a single DataFrame with {len(combined_df)} rows and {len(combined_df.columns)} columns.")
    print("First 5 rows of the combined DataFrame:")
    print(combined_df.head())
else:
    print("No dataframes were loaded. 'combined_df' is not created.")

Successfully combined 12 CSV files into a single DataFrame with 4938837 rows and 12 columns.
First 5 rows of the combined DataFrame:
         date              state      district  pincode  bio_age_5_17  \
0  01-03-2025            Haryana  Mahendragarh   123029         280.0   
1  01-03-2025              Bihar     Madhepura   852121         144.0   
2  01-03-2025  Jammu and Kashmir         Punch   185101         643.0   
3  01-03-2025              Bihar       Bhojpur   802158         256.0   
4  01-03-2025         Tamil Nadu       Madurai   625514         271.0   

   bio_age_17_                                        source_file  \
0        577.0  api_data_aadhar_biometric\api_data_aadhar_biom...   
1        369.0  api_data_aadhar_biometric\api_data_aadhar_biom...   
2       1091.0  api_data_aadhar_biometric\api_data_aadhar_biom...   
3        980.0  api_data_aadhar_biometric\api_data_aadhar_biom...   
4        815.0  api_data_aadhar_biometric\api_data_aadhar_biom...   

   demo_age_5

## Analyze Aadhaar Coverage Growth Rate by State

In [ ]:
combined_df['date'] = pd.to_datetime(combined_df['date'], format='%d-%m-%Y', errors='coerce')

age_columns = [
    'bio_age_5_17', 'bio_age_17_', 
    'demo_age_5_17', 'demo_age_17_', 
    'age_0_5', 'age_5_17', 'age_18_greater',
    'age_group_0_5', 'age_group_5_17', 'age_group_18_over'
]

existing_age_columns = [col for col in age_columns if col in combined_df.columns]

combined_df[existing_age_columns] = combined_df[existing_age_columns].fillna(0)

combined_df['total_activity'] = combined_df[existing_age_columns].sum(axis=1)

state_date_activity = combined_df.groupby(['state', 'date'])['total_activity'].sum().reset_index()

state_date_activity = state_date_activity.sort_values(by=['state', 'date'])

state_date_activity['cumulative_coverage'] = state_date_activity.groupby('state')['total_activity'].cumsum()

state_date_activity['growth_rate'] = state_date_activity.groupby('state')['cumulative_coverage'].pct_change()

print("Date column converted, total_activity, cumulative_coverage, and growth_rate calculated.")
print("First 5 rows of the state_date_activity DataFrame:")
print(state_date_activity.head())

Date column converted, total_activity, cumulative_coverage, and growth_rate calculated.
First 5 rows of the state_date_activity DataFrame:
    state       date  total_activity  cumulative_coverage  growth_rate
0  100000 2025-09-02             3.0                  3.0          NaN
1  100000 2025-09-03             1.0                  4.0     0.333333
2  100000 2025-09-08             1.0                  5.0     0.250000
3  100000 2025-09-09             1.0                  6.0     0.200000
4  100000 2025-09-11             2.0                  8.0     0.333333
